## Imports

In [1]:
from pathlib import Path
import glob
import re

import pandas as pd
import numpy as np

## Configurações

In [2]:
# Diretórios
RAW_DIR = Path("../../data/raw/GSE296007")
PROCESSED_DIR = Path("../../data/interim")

# Arquivos de saída
EXPRESSION_OUTPUT = PROCESSED_DIR / "microplastic_expression.csv"
METADATA_OUTPUT = PROCESSED_DIR / "microplastic_metadata.csv"

# Padrão de entrada
INPUT_PATTERN = str(RAW_DIR / "*.txt.gz")

# Garante que a pasta de saída exista
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Mapeamento de grupos para metadados — FONTE ÚNICA DE VERDADE
# Referenciado por: parse_metadata_from_sample() e derivação de EXPECTED_SAMPLES
GROUP_METADATA_MAP = {
    "CTR": {
        "particle_type": "control",
        "particle_size_um": np.nan,
        "particle_size_nm": np.nan,
        "concentration_gL": 0.0,
        "is_control": True,
        "treatment_status": "control",
    },
    "MA100": {
        "particle_type": "polystyrene",
        "particle_size_um": 0.1,
        "particle_size_nm": 100,
        "concentration_gL": 0.1,
        "is_control": False,
        "treatment_status": "treated",
    },
    "MB100": {
        "particle_type": "polystyrene",
        "particle_size_um": 0.1,
        "particle_size_nm": 100,
        "concentration_gL": 0.01,
        "is_control": False,
        "treatment_status": "treated",
    },
    "MC100": {
        "particle_type": "polystyrene",
        "particle_size_um": 0.1,
        "particle_size_nm": 100,
        "concentration_gL": 0.001,
        "is_control": False,
        "treatment_status": "treated",
    },
    "MA1": {
        "particle_type": "polystyrene",
        "particle_size_um": 1.0,
        "particle_size_nm": 1000,
        "concentration_gL": 0.1,
        "is_control": False,
        "treatment_status": "treated",
    },
    "MB1": {
        "particle_type": "polystyrene",
        "particle_size_um": 1.0,
        "particle_size_nm": 1000,
        "concentration_gL": 0.01,
        "is_control": False,
        "treatment_status": "treated",
    },
    "MC1": {
        "particle_type": "polystyrene",
        "particle_size_um": 1.0,
        "particle_size_nm": 1000,
        "concentration_gL": 0.001,
        "is_control": False,
        "treatment_status": "treated",
    },
    "MD1": {
        "particle_type": "polystyrene",
        "particle_size_um": 1.0,
        "particle_size_nm": 1000,
        "concentration_gL": 0.05,
        "is_control": False,
        "treatment_status": "treated",
    },
}

# Amostras esperadas — derivadas do GROUP_METADATA_MAP (evita duplicação manual)
# Assume 3 réplicas biológicas por grupo (sufixos _1, _2, _3)
EXPECTED_SAMPLES = {
    f"{group}_{rep}"
    for group in GROUP_METADATA_MAP
    for rep in [1, 2, 3]
}

# Threshold mínimo de aproveitamento de leitura (%)
LOW_UTIL_THRESHOLD = 58.0

## Convenção de Nomenclatura das Amostras

Os arquivos de contagem seguem o padrão `GSM{id}_{GRUPO}_{RÉPLICA}_count.txt.gz`.
O grupo codifica o tipo de tratamento:

| Grupo  | Partícula    | Tamanho  | Concentração |
|--------|--------------|----------|--------------|
| CTR    | controle     | —        | 0 g/L        |
| MA100  | poliestireno | 0.1 µm   | 0.1 g/L      |
| MB100  | poliestireno | 0.1 µm   | 0.01 g/L     |
| MC100  | poliestireno | 0.1 µm   | 0.001 g/L    |
| MA1    | poliestireno | 1 µm     | 0.1 g/L      |
| MB1    | poliestireno | 1 µm     | 0.01 g/L     |
| MC1    | poliestireno | 1 µm     | 0.001 g/L    |
| MD1    | poliestireno | 1 µm     | 0.05 g/L     |

O sufixo final (_1, _2, _3) representa a réplica biológica.

## Funções Auxiliares

In [3]:
def parse_sample_name(filename: str) -> str:
    """
    Extrai o nome lógico da amostra a partir do nome do arquivo.

    Exemplo:
        GSM8963286_CTR_1_count.txt.gz -> CTR_1
    """
    base = Path(filename).name
    parts = base.split("_")

    if len(parts) >= 3:
        return f"{parts[1]}_{parts[2]}"

    return Path(base).stem


def load_count_file(filepath: str) -> tuple[pd.Series, float]:
    """
    Lê um arquivo .txt.gz, calcula aproveitamento de leitura e retorna
    (Series por gene_id, porcentagem de aproveitamento).

    Retorna:
        counts   : pd.Series com contagens por gene (linhas técnicas removidas)
        pct_util : float — % de reads mapeados em genes (vs. reads técnicos HTSeq)
    """
    df = pd.read_csv(
        filepath,
        sep="\t",
        header=None,
        names=["gene_id", "count"],
        compression="gzip"
    )

    # Identifica e conta genes técnicos (__no_feature, __ambiguous, __too_low_aQual, __not_aligned, __alignment_not_unique)
    is_tech = df["gene_id"].astype(str).str.startswith("__")
    reads_in_genes = df[~is_tech]["count"].sum()
    reads_technical = df[is_tech]["count"].sum()
    total_reads = reads_in_genes + reads_technical

    # Porcentagem de aproveitamento (retornada — loop controla exibição)
    pct_util = (reads_in_genes / total_reads) * 100 if total_reads > 0 else 0.0

    # Filtra e retorna apenas genes reais
    df_filtered = df[~is_tech]
    counts = df_filtered.set_index("gene_id")["count"]
    return counts, pct_util


def parse_metadata_from_sample(sample_id: str) -> dict:
    """
    Constrói os metadados de uma amostra a partir do sample_id.

    Usa GROUP_METADATA_MAP definido na célula de configuração (fonte única de verdade).
    O sufixo final (_1, _2, _3) representa a réplica biológica.
    """
    match = re.match(r"^([A-Z0-9]+)_([123])$", sample_id)
    if not match:
        raise ValueError(f"Formato de sample_id não reconhecido: {sample_id}")

    group = match.group(1)
    replicate = int(match.group(2))

    if group not in GROUP_METADATA_MAP:
        raise ValueError(f"Grupo não reconhecido no mapeamento: {group}")

    metadata = GROUP_METADATA_MAP[group].copy()
    metadata.update({
        "sample_id": sample_id,
        "group": group,
        "replicate": replicate,
    })

    return metadata

## Carregamento das Amostras

In [4]:
# Carrega os arquivos de contagem e constrói a matriz de expressão
files = sorted(glob.glob(INPUT_PATTERN))

if not files:
    raise FileNotFoundError(f"Nenhum arquivo encontrado no padrão: {INPUT_PATTERN}")

sample_counts = {}
gene_sets = {}           # Conjunto de genes por amostra (para verificação de integridade)
utilization_records = {} # Aproveitamento de leitura por amostra

for filepath in files:
    sample_name = parse_sample_name(filepath)
    counts, pct_util = load_count_file(filepath)
    print(f"{sample_name}: {pct_util:.2f}%")
    sample_counts[sample_name] = counts
    gene_sets[sample_name] = set(counts.index)
    utilization_records[sample_name] = pct_util

# --- VERIFICAÇÃO DE IDENTIDADE DOS GENES ---
first_sample = list(gene_sets.keys())[0]
reference_genes = gene_sets[first_sample]
integrity_error = False

for sample, genes in gene_sets.items():
    if genes != reference_genes:
        print(f"ERRO DE INTEGRIDADE: Amostra {sample} possui um conjunto de genes diferente de {first_sample}!")
        print(f"Diferença encontrada: {genes ^ reference_genes}")
        integrity_error = True

if not integrity_error:
    print(f"\nSucesso: Todas as {len(gene_sets)} amostras possuem os mesmos {len(reference_genes)} genes.")
else:
    print("\nAVISO: Foram detectadas discrepâncias nos genes entre as amostras. Verifique os dados brutos.")

# Constrói o DataFrame de expressão a partir do dicionário de contagens
expression_df = pd.DataFrame(sample_counts).reset_index()
expression_df = expression_df.rename(columns={"index": "gene_id"})

print(f"\nArquivos encontrados: {len(files)}")
print(f"Dimensões da matriz: {expression_df.shape[0]} genes x {expression_df.shape[1] - 1} amostras")

# --- RESUMO DE APROVEITAMENTO DE LEITURA ---
print("\n--- Aproveitamento de Leitura por Amostra ---")
util_df = pd.DataFrame.from_dict(
    utilization_records, orient="index", columns=["utilization_pct"]
)
util_df.index.name = "sample_id"
display(util_df.sort_values("utilization_pct"))

low_util = util_df[util_df["utilization_pct"] < LOW_UTIL_THRESHOLD]
if not low_util.empty:
    print(f"\nAVISO: Amostras com aproveitamento abaixo de {LOW_UTIL_THRESHOLD}%:")
    display(low_util)
else:
    print(f"Todas as amostras apresentam aproveitamento ≥ {LOW_UTIL_THRESHOLD}%.")

expression_df.head()

CTR_1: 64.01%
CTR_2: 61.38%
CTR_3: 62.65%
MA1_1: 61.27%
MA1_2: 62.87%
MA1_3: 59.64%
MD1_1: 58.55%
MD1_2: 59.29%
MD1_3: 60.86%
MB1_1: 60.29%
MB1_2: 62.00%
MB1_3: 59.29%
MC1_1: 58.91%
MC1_2: 60.98%
MC1_3: 61.36%
MA100_1: 58.43%
MA100_2: 61.26%
MA100_3: 58.89%
MB100_1: 58.11%
MB100_2: 62.51%
MB100_3: 63.73%
MC100_1: 62.97%
MC100_2: 63.93%
MC100_3: 64.27%

Sucesso: Todas as 24 amostras possuem os mesmos 63241 genes.

Arquivos encontrados: 24
Dimensões da matriz: 63241 genes x 24 amostras

--- Aproveitamento de Leitura por Amostra ---


,utilization_pct
sample_id,
MB100_1,58.109629
MA100_1,58.429198
MD1_1,58.550339
MA100_3,58.893650
MC1_1,58.914919
MB1_3,59.292503
MD1_2,59.292629
MA1_3,59.643705
MB1_1,60.294802


Todas as amostras apresentam aproveitamento ≥ 58.0%.


,gene_id,CTR_1,CTR_2,CTR_3,MA1_1,MA1_2,MA1_3,MD1_1,MD1_2,MD1_3,...,MC1_3,MA100_1,MA100_2,MA100_3,MB100_1,MB100_2,MB100_3,MC100_1,MC100_2,MC100_3
0,ENSG00000000003,357,327,329,276,226,355,276,365,336,...,370,336,285,323,333,233,334,310,358,398
1,ENSG00000000005,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,ENSG00000000419,770,453,733,605,629,716,655,796,969,...,680,721,700,733,710,703,633,727,974,712
3,ENSG00000000457,27,115,31,26,33,35,42,38,47,...,27,30,31,7,22,40,44,41,68,67
4,ENSG00000000460,37,27,7,4,5,36,3,27,0,...,20,29,3,0,17,9,19,12,21,33


In [5]:
# Valida as amostras carregadas

loaded_samples = [col for col in expression_df.columns if col != "gene_id"]
loaded_set = set(loaded_samples)

missing_samples = EXPECTED_SAMPLES - loaded_set
unexpected_samples = loaded_set - EXPECTED_SAMPLES

print("Amostras carregadas:")
print(sorted(loaded_samples))
print(f"\nTotal de amostras carregadas: {len(loaded_samples)}")

if missing_samples:
    print("\nAVISO: Amostras esperadas, mas ausentes:")
    print(sorted(missing_samples))
else:
    print("\nNenhuma amostra esperada está faltando.")

if unexpected_samples:
    print("\nAVISO: Amostras inesperadas encontradas (não estão em EXPECTED_SAMPLES):")
    print(sorted(unexpected_samples))
else:
    print("Nenhuma amostra inesperada encontrada.")

Amostras carregadas:
['CTR_1', 'CTR_2', 'CTR_3', 'MA100_1', 'MA100_2', 'MA100_3', 'MA1_1', 'MA1_2', 'MA1_3', 'MB100_1', 'MB100_2', 'MB100_3', 'MB1_1', 'MB1_2', 'MB1_3', 'MC100_1', 'MC100_2', 'MC100_3', 'MC1_1', 'MC1_2', 'MC1_3', 'MD1_1', 'MD1_2', 'MD1_3']

Total de amostras carregadas: 24

Nenhuma amostra esperada está faltando.
Nenhuma amostra inesperada encontrada.


## Salvamento da Matriz de Expressão

In [6]:
# Salva a matriz de expressão com colunas ordenadas por tamanho e depois concentração
#
# Ordem: CTR → todos 1µm (MA1, MB1, MC1, MD1) → todos 100nm (MA100, MB100, MC100)
# Isso alinha o CSV com a pergunta de pesquisa (comparação de tamanho de partícula)
# e espelha a ordenação do metadata_df (construído na próxima seção).
#
# As letras A/B/C/D nos grupos RNA-seq seguem a nomenclatura dos grupos transcriptômicos do artigo base

def _sample_sort_key(sample_id):
    group, rep = sample_id.rsplit("_", 1)
    is_ctr    = (group == "CTR")
    is_100nm  = group.endswith("100")       # 100nm (0.1µm) → vem depois de 1µm
    conc_ltr  = group[1] if not is_ctr else ""  # A, B, C ou D
    return (not is_ctr, is_100nm, conc_ltr, int(rep))

data_cols = [c for c in expression_df.columns if c != "gene_id"]
ordered_samples = sorted(data_cols, key=_sample_sort_key)
expression_df = expression_df[["gene_id"] + ordered_samples]

expression_df.to_csv(EXPRESSION_OUTPUT, index=False)
print(f"Matriz de expressão salva em: {EXPRESSION_OUTPUT}")
print(f"Ordem das colunas: gene_id → {ordered_samples}")

Matriz de expressão salva em: ../../data/interim/microplastic_expression.csv
Ordem das colunas: gene_id → ['CTR_1', 'CTR_2', 'CTR_3', 'MA1_1', 'MA1_2', 'MA1_3', 'MB1_1', 'MB1_2', 'MB1_3', 'MC1_1', 'MC1_2', 'MC1_3', 'MD1_1', 'MD1_2', 'MD1_3', 'MA100_1', 'MA100_2', 'MA100_3', 'MB100_1', 'MB100_2', 'MB100_3', 'MC100_1', 'MC100_2', 'MC100_3']


## Criação da Tabela de Metadados

In [7]:
# Constrói a tabela de metadados a partir dos sample_ids
metadata_records = [parse_metadata_from_sample(sample_id) for sample_id in loaded_samples]
metadata_df = pd.DataFrame(metadata_records)

# Ordena: CTR primeiro, depois 1µm (tamanho maior → mais biologicamente ativo nos resultados),
# depois 100nm; dentro de cada tamanho, concentração em ordem alfabética (A→B→C→D), depois réplica.
# Essa ordenação espelha a ordem das colunas da matriz de expressão.
metadata_df = metadata_df.sort_values(
    by=["is_control", "particle_size_um", "group", "replicate"],
    ascending=[False, False, True, True],   # is_control desc → ctrl 1º | size desc → 1µm antes 100nm
    na_position="last"                      # NaN (CTR) fica irrelevante — já separado por is_control
).reset_index(drop=True)

print(f"Dimensões da tabela de metadados: {metadata_df.shape[0]} amostras x {metadata_df.shape[1]} colunas")
metadata_df

Dimensões da tabela de metadados: 24 amostras x 9 colunas


,particle_type,particle_size_um,particle_size_nm,concentration_gL,is_control,treatment_status,sample_id,group,replicate
0,control,NaN,NaN,0.000,True,control,CTR_1,CTR,1
1,control,NaN,NaN,0.000,True,control,CTR_2,CTR,2
2,control,NaN,NaN,0.000,True,control,CTR_3,CTR,3
3,polystyrene,1.0,1000.0,0.100,False,treated,MA1_1,MA1,1
4,polystyrene,1.0,1000.0,0.100,False,treated,MA1_2,MA1,2
5,polystyrene,1.0,1000.0,0.100,False,treated,MA1_3,MA1,3
6,polystyrene,1.0,1000.0,0.010,False,treated,MB1_1,MB1,1
7,polystyrene,1.0,1000.0,0.010,False,treated,MB1_2,MB1,2
8,polystyrene,1.0,1000.0,0.010,False,treated,MB1_3,MB1,3
9,polystyrene,1.0,1000.0,0.001,False,treated,MC1_1,MC1,1


In [8]:
# Salva a tabela de metadados

metadata_df.to_csv(METADATA_OUTPUT, index=False)

print(f"Tabela de metadados salva em: {METADATA_OUTPUT}")

Tabela de metadados salva em: ../../data/interim/microplastic_metadata.csv
